## Fine Tunning de modelo para TechChallenge Fase 3 - PosTech FIAP 9IADT

### Instalação de bibliotecas

In [2]:
#instalando dependências para treinamento e merge do modelo
!pip install -q trl liger-kernel transformers peft datasets bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.3/644.3 kB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.1 MB/s eta 0:00:00


In [3]:
# instalando dependências para o hugging face
!pip install -U huggingface_hub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 16.1 MB/s eta 0:00:00


O colab já tem o pytorch instalado então não precisamos instalá-lo aqui. vamos soente utilizar as bibliotecas do ecossistema do Hugging Face.

In [1]:
# montando o drive para salvamento do modelo treinado
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Código copiado do repostiório e adaptado para rodar no Colab devido a restrições de hardware

In [4]:
import json
import os
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Sequence

import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
# from trl import SFTTrainer - atualizando para parar com o erro do param 'tokenizer'
from trl import SFTConfig, SFTTrainer
from transformers.trainer_utils import get_last_checkpoint # para recuperar o último checkpoint em caso de interrupção da execução e poder retormar de onde parou o treino

print("✅ Todos os pacotes foram importados com sucesso!")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo GPU: {torch.cuda.get_device_name(0)}")

✅ Todos os pacotes foram importados com sucesso!
CUDA disponível: True
Dispositivo GPU: Tesla T4


In [5]:
# login no hugging face para salvamento dos checkpoints

In [6]:
from huggingface_hub import login
from google.colab import userdata
login(token=f"{userdata.get('HF_TOKEN')}")  # token com permissão "write" — huggingface.co/settings/tokens

In [7]:
# preparando api para interação com o repo do HF
from huggingface_hub import HfApi, snapshot_download
api = HfApi()


In [8]:
# --- CAMINHOS NO COLAB ---
QAS_TRAIN_PATH = Path("/content/drive/MyDrive/datasets/qas_train_pt_br.json")
# PROTOCOLS_TRAIN_PATH = Path("/content/drive/MyDrive/datasets/protocol_train.json") -- os protocolos do hospital serão utilizados no RAG
OUTPUT_MODEL_DIR = Path("/content/drive/MyDrive/models/hospital_helper")
OUTPUT_TOKENIZER_DIR = Path("/content/drive/MyDrive/models/hospital_helper_tokenizer")

CHECKPOINT_DIR = Path("/content/drive/MyDrive/finetunning/checkpoints") # Pasta LOCAL contendo os checkpoints de treinamento

BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 2048
PROTOCOL_CHUNK_SIZE = 2400

In [9]:
# subindo a pasta de checkpoints do colab para um repositório no HF
# garante que o repo existe (cria se não existir, não faz nada se já existir)
CHECKPOINT_REPO = "fiap-hospital-helper/hospital-helper-checkpoints" # pasta NO REPO DO HF para salvar o último checkpoint (automático no trainer)
api.create_repo(
    repo_id=CHECKPOINT_REPO,
    repo_type="model",
    private=True,
    exist_ok=True,
)

# agora sim o upload
api.upload_folder(
    folder_path=str(get_last_checkpoint(Path(CHECKPOINT_DIR))),
    repo_id=CHECKPOINT_REPO,
    path_in_repo="last-checkpoint",
    repo_type="model",
)

CommitInfo(commit_url='https://huggingface.co/fiap-hospital-helper/hospital-helper-checkpoints/commit/2452169b33fd2654b249229f337e75516a87e02d', commit_message='Upload folder using huggingface_hub', commit_description='', oid='2452169b33fd2654b249229f337e75516a87e02d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/fiap-hospital-helper/hospital-helper-checkpoints', endpoint='https://huggingface.co', repo_type='model', repo_id='fiap-hospital-helper/hospital-helper-checkpoints'), pr_revision=None, pr_num=None)

In [ ]:

# --- FUNÇÕES AUXILIARES IGUAIS AO PROJETO ---
def _normalize_text(value: Any) -> str:
    if not isinstance(value, str):
        return ""
    return re.sub(r"\s+", " ", value).strip()

def _read_json_list(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")
    with path.open("r", encoding="utf-8") as handle:
        data = json.load(handle)
    if not isinstance(data, list):
        raise ValueError(f"Formato inválido em {path}: esperado uma lista de objetos JSON")
    return data

def _join_contexts(contexts: Any) -> str:
    if not isinstance(contexts, list):
        return ""
    cleaned = [_normalize_text(c) for c in contexts if _normalize_text(c)]
    if not cleaned:
        return ""
    if len(cleaned) == 1:
        return cleaned[0]
    return "\n".join(f"[Contexto {i + 1}] {text}" for i, text in enumerate(cleaned))

def _chunk_text(text: str, max_chunk_size: int) -> List[str]:
    normalized = _normalize_text(text)
    if not normalized:
        return []
    if len(normalized) <= max_chunk_size:
        return [normalized]

    chunks = []
    current_chunk = ""
    for sentence in re.split(r"(?<=[.!?])\s+", normalized):
        sentence = sentence.strip()
        if not sentence:
            continue
        candidate = sentence if not current_chunk else f"{current_chunk} {sentence}"
        if len(candidate) <= max_chunk_size:
            current_chunk = candidate
            continue
        if current_chunk:
            chunks.append(current_chunk.strip())
        current_chunk = sentence
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks or [normalized[:max_chunk_size].strip()]

def _build_qa_example(item: Dict[str, Any]) -> Optional[str]:
    question = _normalize_text(item.get("question"))
    answer = _normalize_text(item.get("answer"))
    contexts = _join_contexts(item.get("contexts"))
    if not question or not answer:
        return None

    prompt_lines = [
        "### Instrucao:",
        "Responda em pt-BR usando o contexto clinico fornecido.",
        "",
        "### Entrada:",
        f"Pergunta: {question}",
    ]
    if contexts:
        prompt_lines.extend(["Contexto:", contexts])
    prompt_lines.extend(["", "### Resposta:", answer])
    return "\n".join(prompt_lines)

def _build_protocol_examples(item: Dict[str, Any]) -> List[str]:
    content_text = _normalize_text(item.get("content_text"))
    if not content_text:
        return []
    name = _normalize_text(item.get("name")) or "protocolo_clinico"
    source = _normalize_text(item.get("source")) or "fonte_desconhecida"
    url = _normalize_text(item.get("url"))

    metadata_lines = [
        "### Protocolo clinico",
        f"Nome: {name}",
        f"Fonte: {source}",
    ]
    if url:
        metadata_lines.append(f"URL: {url}")
    metadata_lines.extend(["", "Conteudo:"])

    prefix = "\n".join(metadata_lines)
    return [f"{prefix}\n{chunk}" for chunk in _chunk_text(content_text, PROTOCOL_CHUNK_SIZE)]

# --- BUSCA O ÚLTIMO CHECKPOINT NO REPO DO HUGGING FACE ---
def _fetch_checkpoint_from_hub(repo_id: str) -> Optional[str]:
    """Tenta baixar o checkpoint mais recente do Hub. Retorna None se não existir ainda."""
    try:
        downloaded_path = snapshot_download(
            repo_id=repo_id,
            allow_patterns=["last-checkpoint/*"],
        )
        ckpt_path = Path(downloaded_path) / "last-checkpoint"
        if ckpt_path.exists() and any(ckpt_path.iterdir()):
            print(f"Checkpoint remoto encontrado em {ckpt_path}")
            return str(ckpt_path)
    except Exception as e:
        print(f"Nenhum checkpoint remoto disponível ainda: {e}")
    return None


# --- EXECUÇÃO DE FINE-TUNING ---
def run_fine_tuning_colab(
    use_4bit: bool = True,
    num_train_epochs: float = 1.0,
    per_device_train_batch_size: int = 1,
    gradient_accumulation_steps: int = 4,
    learning_rate: float = 2e-4,
):
    print("1. Preparando textos de treinamento...")
    qas_data = _read_json_list(QAS_TRAIN_PATH)
    ## protocols_data = _read_json_list(PROTOCOLS_TRAIN_PATH) -- or protocolos médicos serao usados no RAG

    training_texts = []
    for item in qas_data:
        ex = _build_qa_example(item)
        if ex:
            training_texts.append(ex)
    # for item in protocols_data:
    #    training_texts.extend(_build_protocol_examples(item))  -- or protocolos médicos serao usados no RAG

    print(f"Total de exemplos: {len(training_texts)}")

    print("2. Carregando modelo base e tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    ) if use_4bit else None

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        attn_implementation="sdpa" # mais rápido para T4
    )
    model.config.use_cache = False

    if use_4bit:
        model = prepare_model_for_kbit_training(model)

    print("3. Aplicando LoRA...")
    lora_config = LoraConfig(
        r=16,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, lora_config)

    print("4. Configurando TrainingArguments...")
    # comentado devido a atualização de versao do trl
    #training_args = TrainingArguments(
    #    output_dir="/content/drive/MyDrive/finetunning/checkpoints",
    #   num_train_epochs=num_train_epochs,
    #    per_device_train_batch_size=per_device_train_batch_size,
    #    gradient_accumulation_steps=gradient_accumulation_steps,
    #    learning_rate=learning_rate,
    #    warmup_ratio=0.03,
    #    logging_steps=5,
    #    save_strategy="epoch",
    #    save_total_limit=1,
    #    fp16=not torch.cuda.is_bf16_supported(),
    #    bf16=torch.cuda.is_bf16_supported(),
    #    remove_unused_columns=False,
    #    optim="adamw_torch",
    #    gradient_checkpointing=True,
    #)

    training_args = SFTConfig(
        output_dir=CHECKPOINT_DIR,
        #output_dir="/content/drive/MyDrive/finetunning/checkpoints"
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=learning_rate,
        # warmup_ratio=0.03, -- substituído por warmap steps na nova versao
        warmup_steps=130, # reproduz aproximadamente o mesmo cenário do parâmetro anterior
        logging_steps=5,
        # save_strategy="epoch",
        save_strategy="steps", #salvar checkpoint por steps vai garantir a retomada de onde parou em caso de estouro de limite de processamento
        save_steps=100,
        save_total_limit=2, # para o caso de algum checkpoint corromper, mas também não encher o drive
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        remove_unused_columns=False,
        optim="adamw_torch",
        gradient_checkpointing=True,
        max_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        packing=False,
        # --- push automático pro HF Hub a cada save ---
        push_to_hub=True,
        hub_model_id=CHECKPOINT_REPO,
        hub_strategy="checkpoint",
        hub_private_repo=True,
    )


    print("5. Instanciando SFTTrainer (Estrutura idêntica ao backend)...")
    trainer = SFTTrainer(
        model=model,
        #tokenizer=tokenizer, -- renomeado para processing_class
        processing_class=tokenizer,
        train_dataset=Dataset.from_dict({"text": training_texts}),
        #dataset_text_field="text", -- movido para SFTConfig
        #max_seq_length=MAX_SEQ_LENGTH, -- movido para SFTConfig e renomeado para max_length
        args=training_args,
        #packing=False,
    )

    print("6. Treinando modelo...")
    # busca o checkpoint do HF HUB primeiro, se não houver, tenta buscar do drive local e, caso não encontre, inicia o treinamento do zero
    last_checkpoint = _fetch_checkpoint_from_hub(CHECKPOINT_REPO)

    if last_checkpoint: # se houver checkpoint no HUB recupera o último checkpoint
        print(f"Checkpoint encontrado em {last_checkpoint}")
        trainer.train(resume_from_checkpoint=last_checkpoint)

    elif CHECKPOINT_DIR.exists(): # se não encontrar no hub, recupera o último checkpoint do drive
        print(f"Checkpoint encontrado em {CHECKPOINT_DIR}")
        last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)
        trainer.train(resume_from_checkpoint=last_checkpoint)
    else:
      print("Nenhum checkpoint encontrado. Iniciando treinamento do zero.")
      trainer.train()

    print("7. Salvando modelo e tokenizer...")
    OUTPUT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(OUTPUT_MODEL_DIR)
    tokenizer.save_pretrained(OUTPUT_TOKENIZER_DIR)
    print(f"Treinamento concluído com sucesso! Modelo salvo em {OUTPUT_MODEL_DIR}")

# Executar
run_fine_tuning_colab(use_4bit=True, num_train_epochs=1.0)


1. Preparando textos de treinamento...
Total de exemplos: 17407
2. Carregando modelo base e tokenizer...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Checkpoint remoto encontrado em /root/.cache/huggingface/hub/models--fiap-hospital-helper--hospital-helper-checkpoints/snapshots/2452169b33fd2654b249229f337e75516a87e02d/last-checkpoint
Checkpoint encontrado em /root/.cache/huggingface/hub/models--fiap-hospital-helper--hospital-helper-checkpoints/snapshots/2452169b33fd2654b249229f337e75516a87e02d/last-checkpoint


Step,Training Loss
2605,1.023384
2610,1.126524
2615,1.186810
2620,1.110598
2625,1.235216
2630,0.948613
2635,0.916266
2640,1.166469
2645,0.916164
2650,1.098845


### Publicando o modelo no HuggingFace

Agora que temos o modelo treinado, vamos subi o mesmo no hugging face para disponibilizá-lo via API para uso no projeto.

In [ ]:
# atualização do torchao devido a incompatibilidade com peft default do colab
!pip install -U torchao -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 42.2 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [ ]:
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_DIR = "/content/drive/MyDrive/models/hospital_helper"
TOKENIZER_DIR = "/content/drive/MyDrive/models/hospital_helper_tokenizer"
MERGED_DIR = "/content/drive/MyDrive/models/hospital_helper_merged"

In [ ]:

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model = model.merge_and_unload()

model.save_pretrained(MERGED_DIR)
AutoTokenizer.from_pretrained(TOKENIZER_DIR).save_pretrained(MERGED_DIR)

print("Merge concluído!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merge concluído!


In [ ]:
ORG_NAME = "fiap-hospital-helper"  # nome da org no hugging face
MODEL_REPO = f"{ORG_NAME}/hospital-helper-qwen2.5-1.5b"

In [ ]:
## subindo modelo
api.create_repo(
    repo_id=MODEL_REPO,
    repo_type="model",
    private=False,   # necessario para poder publicar no Space (ZeroGPU)
    exist_ok=True
)

api.upload_folder(folder_path=MERGED_DIR, repo_id=MODEL_REPO, repo_type="model")
print(f"Modelo disponível em: https://huggingface.co/{MODEL_REPO}")

Modelo disponível em: https://huggingface.co/fiap-hospital-helper/hospital-helper-qwen2.5-1.5b


In [ ]:
# tageando versão
api.create_tag(repo_id=MODEL_REPO, repo_type="model", tag="v1.0", revision="main")

In [ ]:
# atalização readme
api.upload_file(
    path_or_fileobj="/content/drive/MyDrive/finetunning/MODEL_README.md",
    path_in_repo="README.md",
    repo_id=MODEL_REPO,
    repo_type="model",
)

CommitInfo(commit_url='https://huggingface.co/fiap-hospital-helper/hospital-helper-qwen2.5-1.5b/commit/18058b296ab60f45c68440a73c06f9bc069e0d18', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='18058b296ab60f45c68440a73c06f9bc069e0d18', pr_url=None, repo_url=RepoUrl('https://huggingface.co/fiap-hospital-helper/hospital-helper-qwen2.5-1.5b', endpoint='https://huggingface.co', repo_type='model', repo_id='fiap-hospital-helper/hospital-helper-qwen2.5-1.5b'), pr_revision=None, pr_num=None)